# Notebook 3 — EDA & Visualization

Notebook này ưu tiên dùng `prepared_invoice_dataset_cleaned.csv`. Nếu file đó chưa tồn tại, notebook sẽ tự đọc lại từ `MolaDatabase.invoice_analytics.json` hoặc bạn có thể chạy Notebook 2 trước.

Mục tiêu:
- thống kê mô tả doanh thu
- phân phối total amount
- doanh thu theo tháng
- top khách hàng
- cơ cấu nhóm dịch vụ
- ma trận tương quan biến số

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()


def resolve_path(*candidates):
    for name in candidates:
        p = BASE_DIR / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Không tìm thấy file nào trong danh sách: {candidates}")


def load_json_dataframe(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return pd.json_normalize(data)


def categorize_service(item_name: str) -> str:
    text = str(item_name).lower()
    if 'kiểm định' in text:
        return 'Inspection'
    if 'giám định' in text:
        return 'Survey'
    if 'hiệu chuẩn' in text:
        return 'Calibration'
    if 'thử nghiệm' in text or 'phân tích' in text:
        return 'Testing'
    return 'Other'


In [ ]:
prepared_cleaned = BASE_DIR / 'prepared_invoice_dataset_cleaned.csv'
analytics_path = resolve_path('MolaDatabase.invoice_analytics.json')

if prepared_cleaned.exists():
    df = pd.read_csv(prepared_cleaned)
    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
    print('Đang dùng file đã làm sạch:', prepared_cleaned.name)
else:
    df = load_json_dataframe(analytics_path).rename(columns={
        'date': 'invoice_date',
        'buyer_name': 'buyer_name',
        'total_revenue': 'total_amount',
        'invoice_status': 'invoice_status',
        'payment_method': 'payment_method'
    })
    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')
    print('Không tìm thấy file cleaned, dùng analytics JSON để EDA.')

print('Shape:', df.shape)
display(df.head())

In [ ]:
# Thống kê mô tả chính
summary_cols = [col for col in ['total_amount', 'subtotal_before_tax', 'total_tax', 'total_quantity', 'avg_unit_price'] if col in df.columns]
display(df[summary_cols].describe().T)

In [ ]:
# Biểu đồ 1: Phân phối giá trị hóa đơn
plt.figure(figsize=(8, 5))
plt.hist(df['total_amount'].dropna(), bins=20)
plt.title('Phân phối giá trị hóa đơn')
plt.xlabel('Total Amount')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 2: Doanh thu theo tháng
if 'month' not in df.columns:
    df['month'] = df['invoice_date'].dt.to_period('M').astype(str)
monthly_revenue = df.groupby('month', as_index=False)['total_amount'].sum().sort_values('month')
display(monthly_revenue)

plt.figure(figsize=(9, 5))
plt.plot(monthly_revenue['month'], monthly_revenue['total_amount'], marker='o')
plt.title('Doanh thu theo tháng')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 3: Top 10 khách hàng theo doanh thu
if 'buyer_name' in df.columns:
    top_customers = (
        df.groupby('buyer_name', as_index=False)['total_amount']
        .sum()
        .sort_values('total_amount', ascending=False)
        .head(10)
    )
    display(top_customers)

    plt.figure(figsize=(10, 6))
    plt.barh(top_customers['buyer_name'], top_customers['total_amount'])
    plt.title('Top 10 khách hàng theo doanh thu')
    plt.xlabel('Total Revenue')
    plt.ylabel('Buyer Name')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Biểu đồ 4: Cơ cấu nhóm dịch vụ
if 'service_category' in df.columns:
    service_counts = df['service_category'].value_counts(dropna=False)
    display(service_counts.to_frame('count'))

    plt.figure(figsize=(7, 7))
    plt.pie(service_counts.values, labels=service_counts.index, autopct='%1.1f%%')
    plt.title('Tỷ trọng nhóm dịch vụ')
    plt.tight_layout()
    plt.show()

In [ ]:
# Biểu đồ 5: Ma trận tương quan số học
numeric_df = df.select_dtypes(include=[np.number]).copy()
if not numeric_df.empty:
    corr = numeric_df.corr(numeric_only=True)
    display(corr)

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect='auto')
    plt.title('Ma trận tương quan')
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar()
    plt.tight_layout()
    plt.show()